# Experiment 007 — Jacobian Residual Repair (JRR)

Проверяем, создаёт ли сильный activation steering большой downstream nonlinear Taylor remainder. Diagnostic/calibration и held-out разделены жёстко.


In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/steering-manifold-repair')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/Nek1tt/steering-manifold-repair.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)
print('cwd:', os.getcwd())


## 0. Восстановить frozen steering direction при необходимости
Это не tuning JRR: в чистом runtime лишь воспроизводится уже валидированный sentiment direction.


In [ ]:
from pathlib import Path
direction = Path('results/sentiment_direction.pt')
if not direction.exists():
    subprocess.run([sys.executable,'scripts/validate_sentiment_baseline.py','--config','configs/baseline_sentiment_gpt2.yaml'], check=True)
else:
    print('Using frozen direction:', direction)


## 1. Unit tests математической части


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_jrr.py','tests/test_inference_followups.py','tests/test_denoiser.py'], check=True)


## 2. Real-model JVP preflight
Numerical safety check: основной JVP сравнивается с независимой central finite difference на GPT-2. При failure дальнейший эксперимент запускать нельзя.


In [ ]:
subprocess.run([sys.executable,'scripts/preflight_jrr.py','--config','configs/jrr_gpt2.yaml'], check=True)


## 3. Stage A — диагностика nonlinear propagation
Используются только calibration prompts; выбирается downstream layer и проверяется scaling remainder.


In [ ]:
subprocess.run([sys.executable,'scripts/run_jrr_diagnostic.py','--config','configs/jrr_gpt2.yaml'], check=True)


In [ ]:
import json, pandas as pd
from IPython.display import display, Image
summary = json.loads(Path('results/jrr/diagnostic_summary.json').read_text())
print(json.dumps(summary, indent=2))
display(pd.read_csv('results/jrr/target_layer_summary.csv'))
display(Image(filename='results/jrr/remainder_scaling.png'))
display(Image(filename='results/jrr/orthogonal_remainder_fraction.png'))
display(Image(filename='results/jrr/orthogonal_remainder_vs_fluency.png'))


## 4. Stage B — exact causal oracle на calibration
Запускается только при положительном diagnostic gate. Скрипт дополнительно проверяет gate сам.


In [ ]:
assert summary['oracle_recommended'], 'Diagnostic gate failed: oracle JRR should not be run.'
subprocess.run([sys.executable,'scripts/run_jrr_oracle.py','--config','configs/jrr_gpt2.yaml','--phase','calibration'], check=True)


In [ ]:
cal = json.loads(Path('results/jrr/oracle_calibration_summary.json').read_text())
print(json.dumps(cal, indent=2))
display(pd.read_csv('results/jrr/oracle_calibration_frontier.csv'))
display(Image(filename='results/jrr/oracle_calibration_pareto.png'))


## 5. Frozen held-out
Запускается только после прохождения calibration. После этой точки tuning запрещён.


In [ ]:
assert cal['go_to_heldout'], 'Calibration gate failed: do not open held-out.'
subprocess.run([sys.executable,'scripts/run_jrr_oracle.py','--config','configs/jrr_gpt2.yaml','--phase','evaluation'], check=True)


In [ ]:
display(pd.read_csv('results/jrr/oracle_evaluation_frontier.csv'))
display(Image(filename='results/jrr/oracle_evaluation_pareto.png'))
print(Path('results/jrr/oracle_evaluation_summary.json').read_text())


## Архив результатов
Negative calibration/held-out outcome также является научным результатом; gates не обходятся ради красивого графика.


In [ ]:
import shutil
archive = shutil.make_archive('/content/jrr_results', 'zip', 'results/jrr')
print('Created:', archive)
